Takes your loaded dataset and produces a new one with suboptimal labels, optionally dependent on the input

In [16]:
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt

# ==== Load ====
filepath = '/home/khain/FSNet/datasets/nonsmooth_nonconvex/socp/random2025_socp_dataset_var100_ineq50_eq50_ex10000'
with open(filepath, 'rb') as f:
    dataset = pickle.load(f)

print(dataset.keys())

dict_keys(['Q', 'p', 'A', 'X', 'G', 'h', 'C', 'd', 'YL', 'YU', 'XL', 'XU', 'Y', 'best_partial'])


In [21]:
filepath2 = "/home/khain/FSNet/datasets/nonsmooth_nonconvex/socp/random2025_socp_dataset_var100_ineq50_eq50_ex10000_maxt0.0_ready"
with open(filepath2, 'rb') as f:
    dataset2 = pickle.load(f)

print(dataset2.keys())

dict_keys(['Q', 'p', 'A', 'X', 'G', 'h', 'C', 'd', 'YL', 'YU', 'XL', 'XU', 'solver_name', 'solver_opts', 'budget_type', 'budget_value', 'Y_subopt', 'solve_time_sec', 'iter_count', 'return_status', 'success', 'obj_value', 'eq_l2', 'ineq_max', 'ineq_l2', 'best_partial', 'Y'])


In [22]:
print(dataset2['solve_time_sec'][0:5], dataset2['iter_count'][0], dataset2['obj_value'][0], dataset2['eq_l2'][0], dataset2['ineq_l2'][0])

[4.3852691650390625, 4.4298601150512695, 3.774005174636841, 4.473572015762329, 3.740814208984375] 130 -3.8956892076819303 1.6243723875911994e-14 2.7670949975799886e-09


In [23]:
# check every datapoint in X and X2
for i in range(dataset['X'].shape[0]):
    if not np.array_equal(dataset['X'][i], dataset2['X'][i]):
        raise ValueError(f"Data point {i} in X and X2 are not equal.")

    # if not np.allclose(dataset['Y'][i], dataset2['Y_subopt'][i], atol=1e-3):
    #     print(f"Data point {i} in Y and Y2 are not equal.")

print(dataset2['Y_subopt'][1])
# print(dataset2['Y'][0])
print(dataset['Y'][2])

[-1.0877574   0.06340546 -1.38280029 -0.98196645 -0.11436062 -0.37349927
  1.71073995  0.52350182  0.53333661  0.57367264  1.91198343 -0.93343682
 -0.76321332 -1.98782677  0.35587128  0.09377437 -1.71494515 -1.6498046
 -0.57625547 -1.41181161 -0.50014523 -0.69729508 -1.75295839 -0.24301653
 -1.65933725  1.8461485  -1.33938975 -1.85633515 -0.051458   -0.5317799
 -1.46429133 -1.4333196   0.22364037 -1.28466371  0.8307149   1.10876192
 -0.70722299 -1.02634821  0.88008288  1.72700058 -0.12820067 -1.639703
  1.01143511  0.47011838  1.1765187  -1.73912314 -0.98771362  1.48733624
 -1.39639369 -1.24386451 -0.64294539  0.34444464  0.67625284  1.93529165
 -0.36951027  1.91280617 -1.43159364  1.2477904   0.34195382  1.82091393
  0.90296679  1.89652287  0.55343189 -0.6076312  -1.52055661  1.37879499
 -0.20916503 -0.42293965  1.27964474  1.29303763 -1.73122904 -1.84990055
  1.55773622 -1.17981001  0.01120695  1.13096185  0.19240988 -1.41323043
  0.68554105  0.36656365  0.68117636  1.98743895  0.440

For tolerance type

In [106]:
# # ==== Load ====
# filepath2 = '/home/khain/FSNet/datasets/nonsmooth_nonconvex/socp/random2025_socp_dataset_var100_ineq50_eq50_ex10000_tol1em1_ready'
# with open(filepath2, 'rb') as f:
#     dataset2 = pickle.load(f)

# print(dataset2.keys())

In [107]:
# # check every datapoint in X and X2
# for i in range(dataset['X'].shape[0]):
#     if not np.array_equal(dataset['X'][i], dataset2['X'][i]):
#         print(f"Data point {i} in X and X2 are not equal.")

#     if not np.array_equal(dataset['Y'][i], dataset2['Y'][i]):
#         print(f"Data point {i} in Y and Y2 are not equal.")

# print(dataset2['Y_subopt'][0])
# print(dataset2['Y'][0])
# print(dataset['Y'][0])

In [24]:
# create random suboptimal solutions, sampling from min, max of each dimension of Y over the dataset
# Y_min = np.min(dataset['Y'], axis=0)
# Y_max = np.max(dataset['Y'], axis=0)
Y_subopt = np.random.uniform(low=-3.0, high=3.0, size=dataset['Y'].shape)
Y = dataset['Y']   # shape (N, n)

# ==== Save new dataset ====
dataset_subopt = dict(dataset2)
dataset_subopt['Y_subopt'] = Y_subopt
dataset_subopt['Y'] = Y

savepath = '/home/khain/FSNet/datasets/nonsmooth_nonconvex/socp/random2025_socp_dataset_var100_ineq50_eq50_ex10000_maxt0.0_ready'

os.makedirs(os.path.dirname(savepath), exist_ok=True)
with open(savepath, 'wb') as f:
    pickle.dump(dataset_subopt, f)

print(f"✅ Saved synthetic dataset: {savepath}")
print("Y:", Y.shape, "| Y_subopt:", Y_subopt.shape)

✅ Saved synthetic dataset: /home/khain/FSNet/datasets/nonsmooth_nonconvex/socp/random2025_socp_dataset_var100_ineq50_eq50_ex10000_maxt0.0_ready
Y: (10000, 100) | Y_subopt: (10000, 100)


For noisy type

In [109]:
# # ==== Config ====
# X = dataset['X']   # shape (N, n)
# Y = dataset['Y']   # shape (N, n)

# subopt_noise = 6.0    # random solver variation
# bias_strength = 6.0   # systematic bias magnitude
# input_dependence = True
# savepath = filepath + '_subopt_noise{}_bias{}'.format(subopt_noise, bias_strength)

# # ==== Synthetic suboptimal data ====
# rng = np.random.default_rng(42)

# # 1) Base random noise
# noise = rng.standard_normal(Y.shape) * subopt_noise

# # 2) Scale noise by input magnitude (optional)
# if input_dependence:
#     noise *= 1.0 + np.mean(np.abs(X), axis=1, keepdims=True)

# # 3) Add small mean shift (asymmetry / bias)
# noise += 0.1 * subopt_noise

# # 4) Add input-correlated directional bias
# bias_dir = rng.standard_normal(Y.shape[1])
# bias_dir /= np.linalg.norm(bias_dir) + 1e-8
# bias = bias_strength * bias_dir * np.sign(np.mean(X, axis=1, keepdims=True))

# # 5) Combine
# Y_subopt = Y - noise - bias

# # sample from some curvy f(x)

In [110]:
# showing local deviation patterns.
rng = np.random.default_rng(42)
idx = rng.integers(len(Y))
plt.figure()
plt.plot(Y[idx], label='True Y', color='C0')
plt.plot(Y_subopt[idx], '--', label='Subopt Y', color='C1', alpha=0.8)
plt.fill_between(range(Y.shape[1]), Y[idx], Y_subopt[idx], color='C1', alpha=0.2)
plt.title(f"Example #{idx} (suboptimal sample)")
plt.xlabel("Dimension")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

# showing overall distribution of differences.
gap = 100 * np.mean(Y_subopt - Y, axis=1) / (np.mean(np.abs(Y), axis=1) + 1e-12)
plt.hist(gap, bins=50, color='C1', alpha=0.8, edgecolor='black')
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title("Signed percentage gap distribution")
plt.xlabel("Signed relative gap (%)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# showing correlation or bias direction.
plt.figure()
plt.scatter(Y.flatten(), Y_subopt.flatten(), s=5, alpha=0.5)
plt.plot([Y.min(), Y.max()], [Y.min(), Y.max()], 'k--', lw=1)
plt.xlabel("True Y")
plt.ylabel("Subopt Y")
plt.title("True vs. Suboptimal solutions")
plt.tight_layout()
plt.show()

# showing spread across dataset or noise levels.
import seaborn as sns
sns.violinplot(y=gap, color='C1', inner='box')
plt.title("Distribution of signed relative gaps")
plt.ylabel("Gap (%)")
plt.tight_layout()
plt.show()

NameError: name 'Y' is not defined

In [ ]:
# # ==== Save new dataset ====
# dataset_subopt = dict(dataset)
# dataset_subopt['Y_subopt'] = Y_subopt

# os.makedirs(os.path.dirname(savepath), exist_ok=True)
# with open(savepath, 'wb') as f:
#     pickle.dump(dataset_subopt, f)

# print(f"✅ Saved synthetic dataset: {savepath}")
# print("Y:", Y.shape, "| Y_subopt:", Y_subopt.shape)